# Price Predictions & Model Analysis
## Demonstrating Model Predictions

This notebook shows how to use the trained models for predictions and analyze their performance.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

print("Imports successful!")

## 1. Load Trained Models and Preprocessors

In [ ]:
# Load models
models = {}
model_files = [
    'random_forest_model.pkl',
    'xgboost_model.pkl',
    'gradient_boosting_model.pkl',
    'adaboost_model.pkl'
]

for model_file in model_files:
    try:
        with open(f'../models/{model_file}', 'rb') as f:
            models[model_file.replace('_model.pkl', '').replace('_', ' ').title()] = pickle.load(f)
        print(f"✓ Loaded {model_file}")
    except:
        print(f"✗ Failed to load {model_file}")

# Load preprocessors
with open('../models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
    print("✓ Loaded scaler")

with open('../models/encoder.pkl', 'rb') as f:
    encoder = pickle.load(f)
    print("✓ Loaded encoder")

## 2. Sample Predictions

In [ ]:
# Create sample data for prediction
sample_cars = pd.DataFrame({
    'Brand': ['Toyota', 'Honda', 'BMW', 'Tesla', 'Ford'],
    'Model': ['Camry', 'Civic', '3 Series', 'Model 3', 'Focus'],
    'Year': [2020, 2019, 2021, 2022, 2018],
    'Mileage': [25000, 45000, 15000, 8000, 60000],
    'Engine Size': [2.5, 1.8, 2.0, 0.0, 1.6],
    'Fuel Type': ['Gasoline', 'Gasoline', 'Gasoline', 'Electric', 'Gasoline'],
    'Transmission': ['Automatic', 'Manual', 'Automatic', 'Automatic', 'Manual'],
    'Condition': ['Used', 'Used', 'Like New', 'New', 'Used']
})

print("Sample cars for prediction:")
print(sample_cars)

In [ ]:
# Prepare data for prediction (encode and scale)
from sklearn.preprocessing import LabelEncoder

sample_processed = sample_cars.copy()

# Encode categorical variables
categorical_cols = ['Brand', 'Model', 'Fuel Type', 'Transmission', 'Condition']
for col in categorical_cols:
    sample_processed[col] = encoder.transform(sample_processed[col])

# Scale numerical variables
sample_scaled = scaler.transform(sample_processed)

print("Processed data ready for prediction")

In [ ]:
# Make predictions with all models
predictions = {}

for model_name, model in models.items():
    pred = model.predict(sample_scaled)
    predictions[model_name] = pred
    print(f"\n{model_name} Predictions:")
    for i, car in enumerate(sample_cars['Brand'].values):
        print(f"  {sample_cars.iloc[i]['Brand']} {sample_cars.iloc[i]['Model']}: ${pred[i]:,.2f}")

In [ ]:
# Create predictions dataframe
pred_df = sample_cars[['Brand', 'Model', 'Year']].copy()
for model_name, preds in predictions.items():
    pred_df[model_name] = preds

# Calculate average prediction
pred_cols = [col for col in pred_df.columns if col not in ['Brand', 'Model', 'Year']]
pred_df['Average Prediction'] = pred_df[pred_cols].mean(axis=1)

print("\nAll Model Predictions:")
print(pred_df.to_string())

## 3. Model Predictions Comparison

In [ ]:
# Visualize predictions comparison
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, (i, row) in enumerate(sample_cars.iterrows()):
    if idx < 6:
        model_preds = [predictions[m][i] for m in predictions.keys()]
        
        axes[idx].bar(predictions.keys(), model_preds, color='steelblue')
        axes[idx].set_title(f"{row['Brand']} {row['Model']} ({row['Year']})")
        axes[idx].set_ylabel('Predicted Price')
        axes[idx].tick_params(axis='x', rotation=45)
        axes[idx].grid(True, alpha=0.3, axis='y')
        
        # Add value labels on bars
        for j, v in enumerate(model_preds):
            axes[idx].text(j, v, f'${v:,.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 4. Model Prediction Statistics

In [ ]:
# Calculate prediction statistics
print("\nPrediction Statistics:")
print("="*60)

for car_idx, (i, row) in enumerate(sample_cars.iterrows()):
    car_preds = [predictions[m][i] for m in predictions.keys()]
    
    print(f"\n{row['Brand']} {row['Model']} ({row['Year']})")
    print(f"  Mean Prediction: ${np.mean(car_preds):,.2f}")
    print(f"  Std Dev: ${np.std(car_preds):,.2f}")
    print(f"  Min: ${np.min(car_preds):,.2f}")
    print(f"  Max: ${np.max(car_preds):,.2f}")
    print(f"  Range: ${np.max(car_preds) - np.min(car_preds):,.2f}")

## 5. Feature Importance Analysis

In [ ]:
# Extract feature importance from tree-based models
feature_names = sample_processed.columns.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, (model_name, model) in enumerate(models.items()):
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        indices = np.argsort(importances)[-10:]
        
        axes[idx].barh(range(len(indices)), importances[indices], color='teal')
        axes[idx].set_yticks(range(len(indices)))
        axes[idx].set_yticklabels([feature_names[i] for i in indices])
        axes[idx].set_xlabel('Importance')
        axes[idx].set_title(f'{model_name} - Feature Importance')
    else:
        axes[idx].text(0.5, 0.5, f'{model_name}\nNo feature importance',
                      ha='center', va='center', transform=axes[idx].transAxes)
        axes[idx].set_xticks([])
        axes[idx].set_yticks([])

plt.tight_layout()
plt.show()

## 6. Model Ensemble (Average Prediction)

In [ ]:
# Create ensemble predictions
ensemble_predictions = {}

for i in range(len(sample_cars)):
    preds = [predictions[m][i] for m in predictions.keys()]
    ensemble_predictions[i] = np.mean(preds)

print("\nEnsemble Predictions (Average of all models):")
print("="*60)

for i, row in sample_cars.iterrows():
    print(f"{row['Brand']} {row['Model']}: ${ensemble_predictions[i]:,.2f}")

In [ ]:
# Compare individual vs ensemble
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All models comparison
car_names = [f"{row['Brand']} {row['Model']}" for i, row in sample_cars.iterrows()]

x = np.arange(len(car_names))
width = 0.15

for idx, model_name in enumerate(predictions.keys()):
    axes[0].bar(x + idx*width, predictions[model_name], width, label=model_name)

axes[0].bar(x + len(predictions)*width, list(ensemble_predictions.values()), width, label='Ensemble', color='gold')
axes[0].set_xlabel('Car')
axes[0].set_ylabel('Predicted Price')
axes[0].set_title('All Models vs Ensemble Predictions')
axes[0].set_xticks(x + width * 2)
axes[0].set_xticklabels(car_names, rotation=45, ha='right')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Ensemble only
axes[1].bar(car_names, list(ensemble_predictions.values()), color='gold')
axes[1].set_xlabel('Car')
axes[1].set_ylabel('Predicted Price')
axes[1].set_title('Ensemble Predictions')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

# Add value labels
for i, v in enumerate(ensemble_predictions.values()):
    axes[1].text(i, v, f'${v:,.0f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()